# `curator-simulate` examples
This notebook only shows `!curator-simulate` CLI commands (no Python). It covers ASE runs, thermo logging, ensemble/Mahalanobis uncertainty, TorchSim, and TorchSim batch MD. Commands are split across lines for readability.

## Notes
- Make sure CURATOR and the sample assets under `../test` are available.
- When no dataset is given and only one model is used, uncertainty backends are disabled.
- Run the `!curator-simulate` cells directly; each will create outputs under its `run_path`.

## 1. ASE MD (basic)
Default ASE Langevin engine with basic thermo logging.

In [ ]:
!curator-simulate \\n  simulator=ase_md \\n  model_path=../test/best_model.ckpt \\n  simulator.init_traj=../test/LiFePO4.traj \\n  simulator.run_kwargs.steps=50 \\n  run_path=./runs/ase_md_basic \\n  device=cpu

## 2. ASE + Thermo Logger
Explicit thermo logger settings (basic energies every step).

In [ ]:
!curator-simulate \\n  simulator=ase_md \\n  model_path=../test/best_model.ckpt \\n  simulator.init_traj=../test/LiFePO4.traj \\n  simulator.run_kwargs.steps=50 \\n  simulator.callbacks.thermo.variables=basic_energies \\n  simulator.callbacks.thermo.interval=1 \\n  simulator.callbacks.thermo.header=true \\n  run_path=./runs/ase_md_thermo \\n  device=cpu

## 3. Ensemble uncertainty
Provide multiple model paths; AutoUncertainty will compute ensemble spread (e.g., `force_sd`/`energy_sd`). If you only have one model, you can duplicate the same path to mimic an ensemble.

In [ ]:
!curator-simulate \\n  simulator=auto_uncertainty \\n  model_path=[../test/best_model.ckpt,../test/best_model.ckpt] \\n  simulator.init_traj=../test/LiFePO4.traj \\n  dataset=../test/torchsim_uncertainty/dataset_small.traj \\n  simulator.callbacks.thermo.monitor=force_sd \\n  simulator.callbacks.thermo.uncertainty_backend.ensemble_kwargs.uncertainty_keys=[force_sd,energy_sd] \\n  simulator.callbacks.thermo.save_path=./runs/ensemble_uncertainty/warning_struct.traj \\n  simulator.run_kwargs.steps=30 \\n  run_path=./runs/ensemble_uncertainty \\n  device=cpu

## 4. Mahalanobis uncertainty
Use a reference `dataset` to enable Mahalanobis distance out-of-distribution checks and save high-uncertainty frames.

In [ ]:
!curator-simulate \\n  simulator=auto_uncertainty \\n  model_path=../test/best_model.ckpt \\n  simulator.init_traj=../test/LiFePO4.traj \\n  dataset=../test/torchsim_uncertainty/dataset_small.traj \\n  simulator.callbacks.thermo.monitor=maha_dist \\n  simulator.callbacks.thermo.uncertainty_backend.maha_kwargs.low_threshold=0.9 \\n  simulator.callbacks.thermo.uncertainty_backend.maha_kwargs.high_threshold=0.99 \\n  simulator.callbacks.thermo.save_path=./runs/maha_uncertainty/warning_struct.traj \\n  simulator.run_kwargs.steps=30 \\n  run_path=./runs/maha_uncertainty \\n  device=cpu

## 5. TorchSim
TorchSim is a pure PyTorch MD backend suited for fast, large/batched simulations on GPU. This uses the default TorchSim engine and logger.

In [ ]:
!curator-simulate \\n  simulator=torchsim \\n  model_path=../test/best_model.ckpt \\n  simulator.init_traj=../test/LiFePO4.traj \\n  dataset=../test/torchsim_uncertainty/dataset_small.traj \\n  simulator.run_kwargs.steps=50 \\n  run_path=./runs/torchsim \\n  device=cpu

## 6. TorchSim Batch MD
Run multiple start frames (or multiple systems) in batch by passing a list to `start_index`.

In [ ]:
!curator-simulate \\n  simulator=torchsim \\n  model_path=../test/best_model.ckpt \\n  simulator.init_traj=../test/LiFePO4.traj \\n  simulator.start_index=[0,1] \\n  dataset=../test/torchsim_uncertainty/dataset_small.traj \\n  simulator.run_kwargs.steps=20 \\n  run_path=./runs/torchsim_batch \\n  device=cpu